# 🏠 3D Home Scanner
촬영한 영상을 업로드하면 자동으로 3D 구조도를 만들어드립니다.

**순서:** 셀을 위에서 아래로 순서대로 실행하세요 (Shift+Enter)

In [ ]:
#@title ① Google Drive 연결
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

# 저장 폴더 (Drive 안에 자동 생성)
DRIVE_SAVE_DIR = Path('/content/drive/MyDrive/3D-Scans')
DRIVE_SAVE_DIR.mkdir(parents=True, exist_ok=True)

print(f'✅ Google Drive 연결 완료')
print(f'📂 결과 저장 위치: {DRIVE_SAVE_DIR}')

# 기존 스캔 목록 출력
existing = sorted(DRIVE_SAVE_DIR.glob('*/'), reverse=True)
if existing:
    print(f'\n📋 기존 저장된 스캔 ({len(existing)}개):')
    for d in existing[:5]:
        files = list(d.glob('*.splat')) + list(d.glob('*.ply'))
        label = '✅' if files else '🕐'
        print(f'  {label} {d.name}')
else:
    print('\n(저장된 스캔 없음)')

In [ ]:
#@title ② 환경 설치 (처음 한 번만)
import subprocess, sys

result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
if result.returncode == 0:
    print(f'✅ GPU: {result.stdout.strip()}')
else:
    print('⚠️  GPU 없음. 런타임 > 런타임 유형 변경 > T4 GPU 선택 후 다시 실행하세요.')
    sys.exit()

print('\n📦 NeRFStudio 설치 중... (3~5분 소요)')
subprocess.run(['pip', 'install', 'nerfstudio', '-q'], check=True)
print('✅ 설치 완료!')

In [ ]:
#@title ③ 영상 업로드
from google.colab import files
from pathlib import Path
import shutil, datetime

# 스캔 ID (날짜+시간 기반)
SCAN_ID   = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
WORK_DIR  = Path(f'/content/scan_{SCAN_ID}')
WORK_DIR.mkdir(parents=True)

print('📁 영상 파일을 선택해주세요 (mp4, mov, avi)')
uploaded = files.upload()

video_path = None
for name, data in uploaded.items():
    video_path = WORK_DIR / name
    video_path.write_bytes(data)
    size_mb = len(data) / 1024 / 1024
    print(f'\n✅ 업로드 완료: {name} ({size_mb:.1f} MB)')
    print(f'🗂  스캔 ID: {SCAN_ID}')
    break

In [ ]:
#@title ④ 프레임 추출
import cv2, numpy as np
from IPython.display import display, Image as IPImage

FRAME_INTERVAL_SEC = 0.5  #@param {type:"number"}
MAX_WIDTH = 1280           #@param {type:"integer"}

images_dir = WORK_DIR / 'images'
images_dir.mkdir(exist_ok=True)

cap = cv2.VideoCapture(str(video_path))
fps          = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
duration     = total_frames / fps
frame_step   = max(1, int(fps * FRAME_INTERVAL_SEC))

print(f'🎬 영상: {duration:.1f}초, {fps:.1f}fps')
print(f'📸 {FRAME_INTERVAL_SEC}초마다 추출 → 약 {int(duration/FRAME_INTERVAL_SEC)}장 예상')

saved, idx, previews = 0, 0, []
while True:
    ret, frame = cap.read()
    if not ret: break
    if idx % frame_step == 0:
        h, w = frame.shape[:2]
        if w > MAX_WIDTH:
            frame = cv2.resize(frame, (MAX_WIDTH, int(h * MAX_WIDTH / w)))
        cv2.imwrite(str(images_dir / f'frame_{saved:04d}.jpg'), frame, [cv2.IMWRITE_JPEG_QUALITY, 90])
        if saved < 4: previews.append(frame)
        saved += 1
    idx += 1
cap.release()

print(f'\n✅ {saved}장 추출 완료')
if previews:
    preview = np.hstack([cv2.resize(f, (320, 180)) for f in previews])
    _, buf = cv2.imencode('.jpg', preview)
    display(IPImage(data=buf.tobytes()))
if saved < 20:
    print(f'\n⚠️  프레임 {saved}장 — 10초 이상 촬영을 권장합니다.')

In [ ]:
#@title ⑤ 카메라 포즈 추정 (COLMAP)
import subprocess, json

colmap_dir = WORK_DIR / 'colmap'
colmap_dir.mkdir(exist_ok=True)
print('📐 카메라 포즈 추정 중... (1~3분 소요)')

result = subprocess.run(
    ['ns-process-data', 'images', '--data', str(images_dir), '--output-dir', str(colmap_dir)],
    capture_output=True, text=True
)

if result.returncode != 0 or not (colmap_dir / 'transforms.json').exists():
    print('❌ COLMAP 실패\n', result.stdout[-800:], result.stderr[-400:])
else:
    tf = json.loads((colmap_dir / 'transforms.json').read_text())
    print(f'✅ {len(tf["frames"])}개 프레임 포즈 추정 성공')

In [ ]:
#@title ⑥ Gaussian Splatting 학습
import subprocess

train_dir = WORK_DIR / 'output'
NUM_ITERATIONS = 7000  #@param {type:"integer"}

print(f'🧊 학습 시작 ({NUM_ITERATIONS} iterations) — T4 기준 약 10~15분')

result = subprocess.run(
    [
        'ns-train', 'splatfacto',
        '--data', str(colmap_dir),
        '--output-dir', str(train_dir),
        f'--max-num-iterations={NUM_ITERATIONS}',
        '--viewer.quit-on-train-completion=True',
        '--pipeline.model.cull-alpha-thresh=0.005',
    ],
    capture_output=True, text=True
)

if result.returncode != 0:
    print('❌ 학습 실패\n', result.stdout[-1000:], result.stderr[-500:])
else:
    config_files = list(train_dir.glob('**/config.yml'))
    CONFIG_PATH  = config_files[-1]
    print(f'✅ 학습 완료!')

In [ ]:
#@title ⑦ Export → Google Drive 저장
import subprocess, shutil
from pathlib import Path

export_dir = WORK_DIR / 'export'
export_dir.mkdir(exist_ok=True)

result = subprocess.run(
    ['ns-export', 'gaussian-splat', '--load-config', str(CONFIG_PATH), '--output-dir', str(export_dir)],
    capture_output=True, text=True
)

# 결과 파일 탐색
splat_files = list(export_dir.glob('**/*.splat'))
ply_files   = list(export_dir.glob('**/*.ply'))
SPLAT_PATH  = (splat_files or ply_files or [None])[0]

if SPLAT_PATH is None:
    print('❌ export 실패\n', result.stdout[-500:], result.stderr[-300:])
else:
    size_mb = SPLAT_PATH.stat().st_size / 1024 / 1024
    print(f'✅ {SPLAT_PATH.suffix} 파일 생성 ({size_mb:.1f} MB)')

    # ── Google Drive에 저장 ──────────────────────────────────
    drive_scan_dir = DRIVE_SAVE_DIR / SCAN_ID
    drive_scan_dir.mkdir(parents=True, exist_ok=True)

    dest = drive_scan_dir / SPLAT_PATH.name
    shutil.copy(SPLAT_PATH, dest)
    print(f'\n💾 Google Drive 저장 완료!')
    print(f'   📂 MyDrive/3D-Scans/{SCAN_ID}/{SPLAT_PATH.name}')
    print(f'   📦 {size_mb:.1f} MB')
    DRIVE_SPLAT_PATH = dest

In [ ]:
#@title ⑧ 3D 뷰어
import base64
from IPython.display import HTML

splat_b64 = base64.b64encode(SPLAT_PATH.read_bytes()).decode()
file_ext  = SPLAT_PATH.suffix.lower()

html = f"""
<div style="width:100%;height:600px;background:#08080f;border-radius:16px;overflow:hidden;position:relative;">
  <canvas id="c" style="width:100%;height:100%;display:block;"></canvas>
  <div id="info" style="position:absolute;top:16px;left:50%;transform:translateX(-50%);background:rgba(0,0,0,.6);color:#fff;padding:8px 16px;border-radius:20px;font-family:sans-serif;font-size:13px;">로딩 중…</div>
  <div style="position:absolute;bottom:16px;left:50%;transform:translateX(-50%);color:rgba(255,255,255,.4);font-size:11px;font-family:sans-serif;">드래그: 회전 &nbsp;|&nbsp; 휠: 확대/축소</div>
  <div style="position:absolute;top:16px;right:16px;background:rgba(99,102,241,.8);color:#fff;padding:4px 10px;border-radius:8px;font-size:11px;font-family:sans-serif;">💾 Drive 저장됨</div>
</div>
<script type="importmap">
{{"imports":{{"three":"https://unpkg.com/three@0.160.0/build/three.module.js","three/addons/":"https://unpkg.com/three@0.160.0/examples/jsm/"}}}}
</script>
<script type="module">
import * as THREE from 'three';
import {{ OrbitControls }} from 'three/addons/controls/OrbitControls.js';
import {{ PLYLoader }} from 'three/addons/loaders/PLYLoader.js';

const canvas = document.getElementById('c');
const info   = document.getElementById('info');
const renderer = new THREE.WebGLRenderer({{canvas, antialias:true}});
renderer.setPixelRatio(window.devicePixelRatio);
renderer.setClearColor(0x08080f);
const scene  = new THREE.Scene();
const camera = new THREE.PerspectiveCamera(60, 1, 0.001, 1000);
camera.position.set(0, 0, 3);
const controls = new OrbitControls(camera, canvas);
controls.enableDamping = true;

function resize() {{
  const w = canvas.clientWidth, h = canvas.clientHeight;
  renderer.setSize(w, h, false);
  camera.aspect = w / h;
  camera.updateProjectionMatrix();
}}
resize(); new ResizeObserver(resize).observe(canvas);

const b64 = '{splat_b64}';
const bin = atob(b64);
const bytes = new Uint8Array(bin.length);
for (let i = 0; i < bin.length; i++) bytes[i] = bin.charCodeAt(i);

function centerAndFit(geo) {{
  geo.computeBoundingBox();
  const center = new THREE.Vector3(), size = new THREE.Vector3();
  geo.boundingBox.getCenter(center);
  geo.boundingBox.getSize(size);
  return {{ center, maxDim: Math.max(size.x, size.y, size.z) }};
}}

if ('{file_ext}' === '.ply') {{
  const geo = new PLYLoader().parse(bytes.buffer);
  const mat = new THREE.PointsMaterial({{ size: 0.005, vertexColors: !!geo.attributes.color }});
  if (!geo.attributes.color) mat.color.set(0x8888ff);
  const pts = new THREE.Points(geo, mat);
  const {{ center, maxDim }} = centerAndFit(geo);
  pts.position.sub(center);
  camera.position.set(0, 0, maxDim * 1.5);
  controls.update();
  scene.add(pts);
  info.textContent = `✅ ${{(geo.attributes.position.count / 1000).toFixed(0)}}K 포인트`;
}} else {{
  const ITEM = 32, n = Math.floor(bytes.length / ITEM);
  info.textContent = `파싱 중… ${{(n/1000).toFixed(0)}}K splats`;
  const pos = new Float32Array(n*3), col = new Float32Array(n*3);
  const dv  = new DataView(bytes.buffer);
  for (let i = 0; i < n; i++) {{
    const b = i * ITEM;
    pos[i*3]   = dv.getFloat32(b,    true);
    pos[i*3+1] = dv.getFloat32(b+4,  true);
    pos[i*3+2] = dv.getFloat32(b+8,  true);
    col[i*3]   = bytes[b+24] / 255;
    col[i*3+1] = bytes[b+25] / 255;
    col[i*3+2] = bytes[b+26] / 255;
  }}
  const geo = new THREE.BufferGeometry();
  geo.setAttribute('position', new THREE.BufferAttribute(pos, 3));
  geo.setAttribute('color',    new THREE.BufferAttribute(col, 3));
  const mat = new THREE.PointsMaterial({{ size: 0.008, vertexColors: true, sizeAttenuation: true }});
  const pts = new THREE.Points(geo, mat);
  const {{ center, maxDim }} = centerAndFit(geo);
  pts.position.sub(center);
  camera.position.set(0, 0, maxDim * 1.2);
  controls.update();
  scene.add(pts);
  info.textContent = `✅ ${{(n/1000).toFixed(0)}}K Gaussians`;
}}

(function animate() {{ requestAnimationFrame(animate); controls.update(); renderer.render(scene, camera); }})();
</script>
"""
display(HTML(html))

In [ ]:
#@title ⑨ (선택) Drive에서 이전 스캔 불러와서 보기
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, clear_output

DRIVE_SAVE_DIR = Path('/content/drive/MyDrive/3D-Scans')
scans = sorted(
    [d for d in DRIVE_SAVE_DIR.glob('*/') if list(d.glob('*.splat')) + list(d.glob('*.ply'))],
    reverse=True
)

if not scans:
    print('저장된 스캔이 없습니다.')
else:
    options = {d.name: d for d in scans}
    dropdown = widgets.Dropdown(options=list(options.keys()), description='스캔 선택:')
    btn = widgets.Button(description='뷰어 열기', button_style='primary')
    out = widgets.Output()

    def on_click(_):
        with out:
            clear_output()
            scan_dir = options[dropdown.value]
            f = (list(scan_dir.glob('*.splat')) + list(scan_dir.glob('*.ply')))[0]
            # 전역 변수 갱신 후 ⑧ 셀 재실행하면 해당 파일로 뷰어 열림
            print(f'선택된 파일: {f}')
            print('위 ⑧번 셀에서 SPLAT_PATH 를 아래 경로로 바꾼 뒤 실행하세요:')
            print(f'  SPLAT_PATH = Path("{f}")')

    btn.on_click(on_click)
    display(widgets.HBox([dropdown, btn]), out)